# Driver Drowsiness Detection — Week 6
### Extended Fine-Tuning — Yawn Detection Model
NTCC | Amity School of Engineering & Technology | June 2026

Eye model is already at 99%+, leaving it alone. Yawn model finished Week 5 at 88.99%, which is decent but the confusion matrix showed it was missing a chunk of real yawns. Trying deeper fine-tuning this week — unfreezing more layers and dropping the learning rate further.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

DRIVE  = '/content/drive/MyDrive/NTCC_Drowsiness_Project'
TRAIN  = f'{DRIVE}/data/train'
RES    = f'{DRIVE}/results'
MODELS = f'{DRIVE}/models'

BATCH = 32
SEED  = 42

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
def load_images(classes, label_map, size):
    X, y = [], []
    for cls in classes:
        path = os.path.join(TRAIN, cls)
        imgs = [f for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png'))]
        for fname in imgs:
            img = cv2.imread(os.path.join(path, fname))
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (size, size))
            X.append(img)
            y.append(label_map[cls])
        print(f'  {cls}: {len(imgs)} loaded')
    X = preprocess_input(np.array(X, dtype='float32'))
    y = np.array(y, dtype='int32')
    return X, y

X_face, y_face = load_images(['yawn','no_yawn'], {'yawn':0,'no_yawn':1}, 96)

Xf_tr, Xf_tmp, yf_tr, yf_tmp = train_test_split(X_face, y_face, test_size=0.30, random_state=SEED, stratify=y_face)
Xf_val, Xf_te, yf_val, yf_te = train_test_split(Xf_tmp, yf_tmp, test_size=0.50, random_state=SEED, stratify=yf_tmp)

yf_tr_cat  = to_categorical(yf_tr, 2)
yf_val_cat = to_categorical(yf_val, 2)
yf_te_cat  = to_categorical(yf_te, 2)

print(f'Train: {Xf_tr.shape[0]}  Val: {Xf_val.shape[0]}  Test: {Xf_te.shape[0]}')

In [ ]:
face_model = load_model(f'{MODELS}/face_model.h5')
base_loss, base_acc = face_model.evaluate(Xf_te, yf_te_cat, verbose=0)
print(f'Loaded Week 5 model — current test accuracy: {base_acc*100:.2f}%')

Unfreezing the last 50 layers of MobileNetV2 this time instead of 30, and using a lower learning rate (3e-5 instead of 1e-4) so the extra unfrozen layers don't get pushed around too fast and destroy what was already learned.

In [ ]:
base = face_model.layers[0]
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

face_model.compile(
    optimizer = Adam(learning_rate=3e-5),
    loss      = 'categorical_crossentropy',
    metrics   = ['accuracy']
)

trainable = sum(1 for l in face_model.layers if l.trainable)
print(f'Trainable layers now: {trainable}')

In [ ]:
aug = ImageDataGenerator(rotation_range=8, horizontal_flip=True, brightness_range=[0.8,1.2], zoom_range=0.1)
train_gen = aug.flow(Xf_tr, yf_tr_cat, batch_size=BATCH, shuffle=True)
val_gen   = ImageDataGenerator().flow(Xf_val, yf_val_cat, batch_size=BATCH)

callbacks = [
    ModelCheckpoint(f'{MODELS}/face_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-9, verbose=1),
    CSVLogger(f'{RES}/face_finetune_w6.csv')
]

print('Fine-tuning yawn model...')
history = face_model.fit(
    train_gen,
    steps_per_epoch  = len(Xf_tr) // BATCH,
    epochs           = 25,
    validation_data  = val_gen,
    validation_steps = len(Xf_val) // BATCH,
    callbacks        = callbacks,
    verbose          = 1
)
print(f'Best val_accuracy: {max(history.history["val_accuracy"]):.4f}')

In [ ]:
best_face = load_model(f'{MODELS}/face_model.h5')
loss, acc = best_face.evaluate(Xf_te, yf_te_cat, verbose=0)
print(f'Yawn Model — Test Accuracy: {acc*100:.2f}%')
print(f'Yawn Model — Test Loss    : {loss:.4f}')
print(f'Improvement over Week 5  : {(acc-base_acc)*100:+.2f} points')

In [ ]:
log = pd.read_csv(f'{RES}/face_finetune_w6.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Yawn Model — Week 6 Fine-tuning', fontsize=12)

axes[0].plot(log['epoch']+1, log['accuracy'], label='Train', color='#43A047', lw=2)
axes[0].plot(log['epoch']+1, log['val_accuracy'], label='Val', color='#43A047', lw=2, linestyle='--')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(log['epoch']+1, log['loss'], label='Train', color='#43A047', lw=2)
axes[1].plot(log['epoch']+1, log['val_loss'], label='Val', color='#43A047', lw=2, linestyle='--')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RES}/Yawn_detection_model_accuracy_graph.png', dpi=150)
plt.show()

In [ ]:
yf_pred = np.argmax(best_face.predict(Xf_te, verbose=0), axis=1)
yf_true = np.argmax(yf_te_cat, axis=1)

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(yf_true, yf_pred), display_labels=['yawn','no_yawn']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Yawn Detection — Week 6 Fine-tuned')
plt.tight_layout()
plt.savefig(f'{RES}/Yawn_Detection_confusion_matrix_w6.png', dpi=150)
plt.show()

print(classification_report(yf_true, yf_pred, target_names=['yawn','no_yawn'], digits=4))

In [ ]:
print('='*50)
print('WEEK 6 SUMMARY')
print('='*50)
print(f'Yawn model fine-tuning : 50 layers unfrozen, lr=3e-5')
print(f'Eye  model accuracy    : 99.08% (unchanged)')
print(f'Yawn model accuracy    : {acc*100:.2f}% (improved from {base_acc*100:.2f}%)')
print()
print('Both models now finalised. Saved to Google Drive:')
for f in ['eye_model.h5', 'face_model.h5']:
    path = f'{MODELS}/{f}'
    if os.path.exists(path):
        print(f'  {f} — {os.path.getsize(path)/1e6:.1f} MB')
print()
print('Next (Week 7): real-time webcam integration')
print('='*50)